|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Quantization<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: quantize it, then find the speed you lost<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import torch
import cudalib

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

Quantize the weights, then find out why it did not make anything faster.

Stages 18 and 18b. The first half is arithmetic and the second half is a
measurement that should annoy you.

# Exercise 1: INT8, per output channel

In [ ]:
def quantize_int8(weight):
  """weight (out, in) -> (int8_weight, float scales (out,)). Symmetric, one
  scale for each OUTPUT channel. Keep the scale above zero: an all-zero
  row otherwise gives inf."""
  scales = weight.abs().amax(dim=1).clamp(min=1e-8) / 127.0
  int8_weight = torch.round(weight / scales[:, None]).clamp(-127, 127).to(torch.int8)
  return int8_weight, scales.float()

def dequantize_int8(int8_weight, scales):
  return int8_weight.float() * scales[:, None]

def relative_error(approximate, exact):
  """The mean absolute error, relative to the mean absolute value."""
  return ((approximate - exact).abs().mean() / exact.abs().mean()).item()

weight = torch.randn(512, 1024, device=device)
int8_weight, scales = quantize_int8(weight)
assert int8_weight.dtype == torch.int8 and scales.shape == (512,)
print(f'mean relative error {relative_error(dequantize_int8(int8_weight, scales), weight):.4f}')
print(f'bytes: {weight.numel()*4/1e6:.2f} MB fp32 -> '
      f'{(int8_weight.numel() + scales.numel()*4)/1e6:.2f} MB')

# Exercise 2: why per channel and not per tensor

In [ ]:
# One row with very large weights, as a real model has.
outlier_weight = weight.clone()
outlier_weight[0] *= 100.0
scale = outlier_weight.abs().max() / 127.0
per_tensor_error = relative_error(
    torch.round(outlier_weight / scale).clamp(-127, 127) * scale, outlier_weight)
per_channel_error = relative_error(dequantize_int8(*quantize_int8(outlier_weight)),
                                   outlier_weight)
print(f'per tensor  {per_tensor_error:.4f}')
print(f'per channel {per_channel_error:.4f}   ({per_tensor_error/per_channel_error:.0f}x better)')

# Exercise 3: now time it

Half the bytes should be most of half the time. Check.

In [ ]:
if device == 'cuda':
  bf16_weight = torch.randn(4096, 4096, device=device, dtype=torch.bfloat16)
  int8_weight, scales = quantize_int8(bf16_weight.float())
  bf16_scales = scales.to(torch.bfloat16)
  inputs = torch.randn(1, 4096, device=device, dtype=torch.bfloat16)
  bf16_ms = cudalib.bench_ms(lambda: inputs @ bf16_weight.t(), best_of=3)
  unfused_ms = cudalib.bench_ms(
      lambda: inputs @ (int8_weight.to(torch.bfloat16) * bf16_scales[:, None]).t(), best_of=3)
  print(f'bf16 matmul:             {bf16_ms:7.3f} ms')
  print(f'dequantize then matmul:  {unfused_ms:7.3f} ms  ({unfused_ms/bf16_ms:.1f}x SLOWER)')
  print(f'\nhalf the weight bytes made it {unfused_ms/bf16_ms:.1f} times slower.')
  print('The rest of stage 18b tells you why.')

### Two results, and the second one is the stage

**A per-channel scale is approximately fifty times more accurate** than a
single tensor scale, on a matrix with one outlier row. Real weight matrices
have outlier rows. A per-channel scale costs one float for each output
channel, which is nothing.

**And the quantization made the multiply slower.** It is not a small
difference. It is several times slower than no quantization at all.

You made the stored bytes half as large, and then you built them again. So the
memory system moved the small weights AND a full-size bf16 copy. The
arithmetic units did the same work as before.

This is the most common way to deploy quantization incorrectly. The symptom is
that the model became smaller and the server became slower. Nobody expects
that, and everybody blames another part of the system.

The fix is the epilogue. The scale is per output channel, so it comes out of
the sum. Apply it one time, to a value that is already in a register, after
the dot product. Stage 18b makes you write that kernel.

    ./vc guide 18b